# NeuralKeyGen — ML-Based Deterministic Key Generation
## Algoritma Enkripsi Kunci Siswa untuk DMS Sekolah

**Apa yang dilakukan notebook ini:**
1. Generate 500.000 data training sintetis
2. Membangun & melatih model NeuralKeyGen (PyTorch)
3. Evaluasi: deterministik, collision-resistance, avalanche effect
4. Export model `.pt` siap pakai di aplikasi FastAPI

> **Runtime yang disarankan:** GPU (T4) — Runtime > Change runtime type > GPU


In [ ]:
# ── Instalasi library tambahan ──────────────────────────────────────────────
!pip install -q pycryptodome matplotlib seaborn tqdm

import os, json, time, secrets, hashlib, random, struct
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from Crypto.Protocol.KDF import HKDF
from Crypto.Hash import SHA256

# Cek GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device   : {device}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch  : {torch.__version__}")
print(f"NumPy    : {np.__version__}")
print("✓ Semua library siap")


## 1. Konfigurasi Global

In [ ]:
# ─── KONFIGURASI — ubah sesuai kebutuhan ────────────────────────────────────

CFG = {
    # Dataset
    "N_SAMPLES"      : 500_000,   # jumlah siswa sintetis
    "FEATURE_DIM"    : 64,        # total dimensi input vector
    "OUTPUT_DIM"     : 256,       # output bits (akan di-derive jadi 32 bytes AES key)
    "TRAIN_RATIO"    : 0.70,
    "VAL_RATIO"      : 0.15,
    # TEST_RATIO      : 0.15 (sisa)

    # Model
    "HIDDEN_1"       : 512,
    "HIDDEN_2"       : 256,
    "DROPOUT"        : 0.3,

    # Training
    "EPOCHS"         : 50,
    "BATCH_SIZE"     : 2048,
    "LR"             : 3e-4,
    "WEIGHT_DECAY"   : 1e-5,
    "PATIENCE"       : 8,         # early stopping

    # Loss weights
    "W_CONSISTENCY"  : 0.50,
    "W_SEPARATION"   : 0.30,
    "W_ENTROPY"      : 0.20,

    # System salt — GANTI dengan nilai acak di produksi!
    "SYSTEM_SALT"    : b"DMS_SEKOLAH_SALT_v1_2025",

    # Output
    "SAVE_DIR"       : "/content/neuralkeygen_output",
    "MODEL_NAME"     : "neuralkeygen.pt",
    "SEED"           : 42,
}

# Reproducibility
random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["SEED"])

Path(CFG["SAVE_DIR"]).mkdir(parents=True, exist_ok=True)
print("✓ Konfigurasi dimuat")
print(f"  Sampel training : {CFG['N_SAMPLES']:,}")
print(f"  Dimensi input   : {CFG['FEATURE_DIM']}")
print(f"  Dimensi output  : {CFG['OUTPUT_DIM']} bit")
print(f"  Epochs          : {CFG['EPOCHS']}")
print(f"  Batch size      : {CFG['BATCH_SIZE']}")
print(f"  Save dir        : {CFG['SAVE_DIR']}")


## 2. Feature Engineering — Cara Mengubah Profil Siswa Jadi Vektor 64D

In [ ]:
# ─── FEATURE EXTRACTOR ──────────────────────────────────────────────────────

class StudentFeatureExtractor:
    """
    Mengubah profil siswa menjadi vektor numerik 64 dimensi.
    
    Breakdown dimensi:
      [0:8]   NIS → sinusoidal positional encoding       (8 dim)
      [8:12]  Tahun masuk → cyclical + linear             (4 dim)
      [12:16] Kode sekolah (NPSN) → hash normalize        (4 dim)
      [16:24] Hash nama → SHA-256 truncate normalize      (8 dim)
      [24:32] Tanggal lahir → cyclical sin/cos            (8 dim)
      [32:48] Entropy seed → float normalize              (16 dim)
      [48:52] Timestamp registrasi → cyclical             (4 dim)
      [52:56] System salt contribution                    (4 dim)
      [56:60] Cross features (NIS x tgl lahir)            (4 dim)
      [60:64] Padding / future use                        (4 dim)
    Total: 64 dimensi
    """

    def __init__(self, system_salt: bytes = CFG["SYSTEM_SALT"]):
        self.salt = system_salt

    def _sinusoidal(self, value: float, n_dims: int, scale: float = 10000.0) -> np.ndarray:
        """Positional encoding ala Transformer."""
        d = n_dims // 2
        enc = np.array([
            np.sin(value / (scale ** (2 * i / n_dims))) for i in range(d)
        ] + [
            np.cos(value / (scale ** (2 * i / n_dims))) for i in range(d)
        ], dtype=np.float32)
        return enc

    def _cyclical(self, value: float, period: float) -> np.ndarray:
        """Cyclical encoding: sin + cos."""
        angle = 2 * np.pi * value / period
        return np.array([np.sin(angle), np.cos(angle)], dtype=np.float32)

    def _hash_to_float(self, text: str, n_bytes: int) -> np.ndarray:
        """SHA-256 hash → float array [0, 1]."""
        h = hashlib.sha256(text.encode('utf-8')).digest()
        return np.frombuffer(h[:n_bytes], dtype=np.uint8).astype(np.float32) / 255.0

    def extract(self, siswa: dict) -> np.ndarray:
        """
        Input dict keys:
          nis, tahun_masuk, npsn, nama_hash, tgl_lahir (YYYY-MM-DD),
          entropy_seed (hex str 32 chars = 16 bytes), reg_timestamp (int unix)
        """
        features = []

        # [0:8] NIS sinusoidal
        nis_val = int(str(siswa['nis']).replace('-', '')) % 100_000_000
        features.append(self._sinusoidal(nis_val, 8, scale=10_000_000))

        # [8:12] Tahun masuk
        tahun = int(siswa['tahun_masuk'])
        features.append(self._cyclical(tahun - 2010, 20))          # sin, cos
        features.append(np.array([(tahun - 2010) / 20.0,
                                    float(tahun % 2)], dtype=np.float32))

        # [12:16] NPSN hash
        features.append(self._hash_to_float(siswa['npsn'], 4))

        # [16:24] Nama hash (bukan nama asli, tapi hash-nya)
        features.append(self._hash_to_float(siswa['nama_hash'], 8))

        # [24:32] Tanggal lahir cyclical
        tgl = datetime.strptime(siswa['tgl_lahir'], '%Y-%m-%d')
        features.append(self._cyclical(tgl.day, 31))
        features.append(self._cyclical(tgl.month, 12))
        features.append(self._cyclical(tgl.year - 1995, 30))
        features.append(np.array([float(tgl.weekday()) / 6.0,
                                    float(tgl.month <= 6)], dtype=np.float32))

        # [32:48] Entropy seed (16 bytes → 16 float)
        seed_bytes = bytes.fromhex(siswa['entropy_seed'])
        features.append(np.frombuffer(seed_bytes, dtype=np.uint8).astype(np.float32) / 255.0)

        # [48:52] Timestamp registrasi cyclical
        ts = int(siswa.get('reg_timestamp', 1_700_000_000))
        features.append(self._cyclical(ts % 86400, 86400))          # detik dalam hari
        features.append(self._cyclical((ts // 86400) % 365, 365))   # hari dalam tahun

        # [52:56] Salt contribution — buat setiap sekolah berbeda
        salt_input = siswa['npsn'] + siswa['entropy_seed'][:8]
        features.append(self._hash_to_float(salt_input, 4))

        # [56:60] Cross features
        cross_val = nis_val * tgl.month * tgl.day
        features.append(self._sinusoidal(cross_val % 1_000_000, 4, scale=100_000))

        # [60:64] Padding zeros (reserved)
        features.append(np.zeros(4, dtype=np.float32))

        vec = np.concatenate(features)
        assert vec.shape[0] == CFG["FEATURE_DIM"], f"Expected 64, got {vec.shape[0]}"
        return vec


# Test ekstraksi
extractor = StudentFeatureExtractor()
test_siswa = {
    'nis': '20240001',
    'tahun_masuk': 2024,
    'npsn': 'NPSN1234',
    'nama_hash': hashlib.sha256('Budi Santoso'.encode()).hexdigest(),
    'tgl_lahir': '2008-03-15',
    'entropy_seed': secrets.token_hex(16),
    'reg_timestamp': int(time.time()),
}
vec = extractor.extract(test_siswa)
print(f"✓ Feature vector shape : {vec.shape}")
print(f"  Min value            : {vec.min():.4f}")
print(f"  Max value            : {vec.max():.4f}")
print(f"  Mean                 : {vec.mean():.4f}")
print(f"  Std                  : {vec.std():.4f}")
print(f"  Sample (first 8)     : {np.round(vec[:8], 3)}")


## 3. Generate Data Training Sintetis (500.000 Siswa)

In [ ]:
# ─── SYNTHETIC DATA GENERATOR ───────────────────────────────────────────────

class SyntheticStudentGenerator:
    """
    Membuat profil siswa sintetis yang realistis.
    TIDAK menggunakan data nyata — aman secara privasi.
    """

    NAMA_DEPAN = [
        'Andi', 'Budi', 'Citra', 'Dewi', 'Eko', 'Fitri', 'Galih', 'Hani',
        'Indra', 'Joko', 'Kartika', 'Lina', 'Muhamad', 'Nadia', 'Otto',
        'Putri', 'Qori', 'Rizki', 'Sari', 'Taufik', 'Ulfa', 'Veri',
        'Wulan', 'Xena', 'Yogi', 'Zahra', 'Arif', 'Bella', 'Chandra',
        'Dian', 'Endang', 'Fajar', 'Gilang', 'Hendra', 'Irma', 'Jihan',
    ]
    NAMA_BELAKANG = [
        'Santoso', 'Wijaya', 'Kusuma', 'Pratama', 'Sari', 'Wibowo',
        'Setiawan', 'Nugroho', 'Hidayat', 'Susanto', 'Rahayu', 'Purnomo',
        'Utama', 'Wahyudi', 'Lestari', 'Saputra', 'Anwar', 'Suryadi',
        'Hartono', 'Budiman', 'Firmansyah', 'Harahap', 'Siregar', 'Nasution',
    ]

    def __init__(self, n_schools: int = 50, seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        # Generate pool NPSN sekolah
        self.npsn_pool = [f"NPSN{i:05d}" for i in range(1, n_schools + 1)]
        self.extractor = StudentFeatureExtractor()

    def generate_one(self, idx: int) -> dict:
        """Generate satu profil siswa sintetis."""
        random.seed(idx * 9973 + 12345)  # Deterministic per idx
        np.random.seed(idx * 7919 + 54321)

        nama_depan = random.choice(self.NAMA_DEPAN)
        nama_belakang = random.choice(self.NAMA_BELAKANG)
        nama_lengkap = f"{nama_depan} {nama_belakang}"

        # Tahun masuk: 2018-2024
        tahun_masuk = random.randint(2018, 2024)

        # NIS: tahun masuk + urut 4 digit
        urut = (idx % 9999) + 1
        nis = f"{tahun_masuk}{urut:04d}"

        # Tanggal lahir: SMA ~ usia 14-18 tahun saat masuk
        usia_masuk = random.randint(14, 16)
        tahun_lahir = tahun_masuk - usia_masuk
        bulan_lahir = random.randint(1, 12)
        hari_max = [31,28,31,30,31,30,31,31,30,31,30,31][bulan_lahir-1]
        hari_lahir = random.randint(1, hari_max)
        tgl_lahir = f"{tahun_lahir}-{bulan_lahir:02d}-{hari_lahir:02d}"

        # NPSN sekolah
        npsn = self.npsn_pool[idx % len(self.npsn_pool)]

        # Entropy seed — acak kriptografis (pakai secrets bukan random)
        # Agar reproducible dalam dataset, kita derive dari idx + UUID namespace
        seed_input = f"STUDENT_SEED_{idx}_{nis}_{tgl_lahir}"
        entropy_seed = hashlib.sha256(seed_input.encode()).hexdigest()[:32]

        # Timestamp registrasi
        base_ts = 1_700_000_000
        reg_timestamp = base_ts + idx * 3600 + random.randint(0, 3599)

        # Hash nama (tidak simpan nama asli ke training data)
        nama_hash = hashlib.sha256(nama_lengkap.lower().encode()).hexdigest()

        return {
            'idx': idx,
            'nis': nis,
            'tahun_masuk': tahun_masuk,
            'npsn': npsn,
            'nama_hash': nama_hash,
            'tgl_lahir': tgl_lahir,
            'entropy_seed': entropy_seed,
            'reg_timestamp': reg_timestamp,
        }

    def generate_dataset(self, n: int) -> tuple:
        """Generate n profil dan return (features_array, metadata_list)."""
        features = np.zeros((n, CFG["FEATURE_DIM"]), dtype=np.float32)
        meta = []

        print(f"Generating {n:,} synthetic students...")
        for i in tqdm(range(n), desc="Generating"):
            siswa = self.generate_one(i)
            features[i] = self.extractor.extract(siswa)
            meta.append(siswa)

        return features, meta


# ─── Generate dataset ────────────────────────────────────────────────────────
gen = SyntheticStudentGenerator(n_schools=50)

start = time.time()
X_all, meta_all = gen.generate_dataset(CFG["N_SAMPLES"])
elapsed = time.time() - start

print(f"\n✓ Dataset berhasil dibuat!")
print(f"  Shape          : {X_all.shape}")
print(f"  Memory         : {X_all.nbytes / 1e6:.1f} MB")
print(f"  Waktu generate : {elapsed:.1f} detik")
print(f"  Sample [0]     : NIS={meta_all[0]['nis']}, NPSN={meta_all[0]['npsn']}")
print(f"  Sample [1]     : NIS={meta_all[1]['nis']}, NPSN={meta_all[1]['npsn']}")

# Cek tidak ada duplikat entropy seed
seeds = [m['entropy_seed'] for m in meta_all]
print(f"  Unique seeds   : {len(set(seeds)):,} / {len(seeds):,} (harus sama)")


## 4. Eksplorasi Data (EDA)

In [ ]:
# ─── EKSPLORASI DATA ─────────────────────────────────────────────────────────

fig = plt.figure(figsize=(16, 10))
fig.suptitle("NeuralKeyGen — Analisis Feature Vector", fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Distribusi nilai per fitur (violin)
ax1 = fig.add_subplot(gs[0, :2])
sample_cols = [0, 8, 16, 24, 32, 48, 56]
sample_data = [X_all[:2000, c] for c in sample_cols]
bp = ax1.violinplot(sample_data, positions=range(len(sample_cols)),
                    showmeans=True, showmedians=True)
for pc in bp['bodies']:
    pc.set_facecolor('#4A90D9')
    pc.set_alpha(0.6)
ax1.set_xticks(range(len(sample_cols)))
ax1.set_xticklabels([f'dim {c}' for c in sample_cols], fontsize=9)
ax1.set_ylabel("Nilai (0-1)")
ax1.set_title("Distribusi nilai pada dimensi representatif")
ax1.grid(True, alpha=0.3)

# 2. Korelasi antar dimensi (heatmap sample)
ax2 = fig.add_subplot(gs[0, 2])
sample_dims = list(range(0, 64, 8))
corr = np.corrcoef(X_all[:5000, sample_dims].T)
im = ax2.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax2.set_xticks(range(len(sample_dims)))
ax2.set_yticks(range(len(sample_dims)))
ax2.set_xticklabels([str(d) for d in sample_dims], fontsize=8)
ax2.set_yticklabels([str(d) for d in sample_dims], fontsize=8)
ax2.set_title("Korelasi antar dimensi")
plt.colorbar(im, ax=ax2, shrink=0.8)

# 3. Mean & std per dimensi
ax3 = fig.add_subplot(gs[1, :2])
means = X_all.mean(axis=0)
stds = X_all.std(axis=0)
dims = np.arange(CFG["FEATURE_DIM"])
ax3.fill_between(dims, means - stds, means + stds, alpha=0.3, color='#4A90D9', label='±1 std')
ax3.plot(dims, means, color='#2563EB', linewidth=1.5, label='Mean')
ax3.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='0.5 target')
ax3.set_xlabel("Dimensi")
ax3.set_ylabel("Nilai")
ax3.set_title("Mean ± Std setiap dimensi (semua sampel)")
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# 4. Distribusi jarak antar vektor acak
ax4 = fig.add_subplot(gs[1, 2])
n_pairs = 2000
idx1 = np.random.choice(len(X_all), n_pairs, replace=False)
idx2 = np.random.choice(len(X_all), n_pairs, replace=False)
dists = np.linalg.norm(X_all[idx1] - X_all[idx2], axis=1)
ax4.hist(dists, bins=50, color='#4A90D9', edgecolor='white', linewidth=0.5)
ax4.axvline(dists.mean(), color='red', linestyle='--', label=f'Mean={dists.mean():.2f}')
ax4.set_xlabel("Jarak Euclidean")
ax4.set_ylabel("Frekuensi")
ax4.set_title("Distribusi jarak antar vektor siswa berbeda")
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

plt.savefig(f"{CFG['SAVE_DIR']}/eda_features.png", dpi=120, bbox_inches='tight')
plt.show()
print("✓ EDA selesai")
print(f"  Global mean : {X_all.mean():.4f} (ideal ~0.5)")
print(f"  Global std  : {X_all.std():.4f}")
print(f"  Min         : {X_all.min():.4f}")
print(f"  Max         : {X_all.max():.4f}")


## 5. PyTorch Dataset — Pasangan untuk Training

In [ ]:
# ─── PYTORCH DATASET ─────────────────────────────────────────────────────────

class NeuralKeyGenDataset(Dataset):
    """
    Dataset berisi TRIPLET untuk training:
      anchor   : vektor siswa A
      positive : vektor siswa A (dengan augmentasi kecil)  → harus output SAMA
      negative : vektor siswa B (berbeda)                  → harus output BERBEDA
    """

    def __init__(self, features: np.ndarray, augment_std: float = 0.001):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.n = len(features)
        self.augment_std = augment_std

    def __len__(self):
        return self.n

    def __getitem__(self, idx: int):
        anchor = self.X[idx]

        # Positive: input yang sama + noise sangat kecil
        # (simulasi: login dari device berbeda, IP berbeda)
        # Model harus output IDENTIK meski ada noise kecil
        noise = torch.randn_like(anchor) * self.augment_std
        positive = torch.clamp(anchor + noise, 0.0, 1.0)

        # Negative: siswa acak yang berbeda
        neg_idx = random.randint(0, self.n - 1)
        while neg_idx == idx:
            neg_idx = random.randint(0, self.n - 1)
        negative = self.X[neg_idx]

        return anchor, positive, negative


# Split dataset
n_total = len(X_all)
n_train = int(n_total * CFG["TRAIN_RATIO"])
n_val   = int(n_total * CFG["VAL_RATIO"])
n_test  = n_total - n_train - n_val

X_train = X_all[:n_train]
X_val   = X_all[n_train:n_train + n_val]
X_test  = X_all[n_train + n_val:]

train_ds = NeuralKeyGenDataset(X_train, augment_std=0.001)
val_ds   = NeuralKeyGenDataset(X_val,   augment_std=0.0)   # no augment saat val
test_ds  = NeuralKeyGenDataset(X_test,  augment_std=0.0)

train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["BATCH_SIZE"] * 2,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["BATCH_SIZE"] * 2,
                          shuffle=False, num_workers=2)

print("✓ Dataset siap:")
print(f"  Train   : {len(train_ds):,} samples  ({len(train_loader)} batches)")
print(f"  Val     : {len(val_ds):,} samples  ({len(val_loader)} batches)")
print(f"  Test    : {len(test_ds):,} samples  ({len(test_loader)} batches)")

# Cek satu batch
a, p, n = next(iter(train_loader))
print(f"\n  Batch shapes: anchor={a.shape}, positive={p.shape}, negative={n.shape}")
print(f"  anchor vs positive diff (mean): {(a - p).abs().mean():.6f}  ← harus kecil")
print(f"  anchor vs negative diff (mean): {(a - n).abs().mean():.4f}  ← harus besar")


## 6. Arsitektur Model NeuralKeyGen

In [ ]:
# ─── MODEL ARSITEKTUR ────────────────────────────────────────────────────────

class NeuralKeyGen(nn.Module):
    """
    Neural network deterministik untuk generate kunci enkripsi 256-bit.

    Properti utama:
    - Deterministik: model.eval() + torch.no_grad() → output selalu sama
    - Collision-resistant: output berbeda sangat jauh untuk input berbeda
    - Entropy tinggi: distribusi output mendekati uniform [0,1]
    """

    def __init__(self, input_dim: int, hidden1: int, hidden2: int,
                 output_dim: int, dropout: float):
        super().__init__()

        self.encoder = nn.Sequential(
            # Blok 1: input → hidden1
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            # Blok 2: hidden1 → hidden2
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            # Blok 3: hidden2 → hidden2 (depth tambahan)
            nn.Linear(hidden2, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
        )

        # Output head: raw vector panjang output_dim
        self.output_head = nn.Sequential(
            nn.Linear(hidden2, output_dim),
            nn.Sigmoid(),   # output ∈ [0, 1] setiap elemen
        )

        # Inisialisasi bobot dengan Kaiming (cocok untuk ReLU)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x)
        out = self.output_head(h)
        return out

    def get_key_bytes(self, x: torch.Tensor,
                       salt: bytes = CFG["SYSTEM_SALT"]) -> bytes:
        """
        Inference: input → raw output → HKDF → 32 bytes AES-256 key.
        Dipanggil saat produksi, bukan saat training.
        """
        self.eval()
        with torch.no_grad():
            if x.dim() == 1:
                x = x.unsqueeze(0)
            raw = self.forward(x).squeeze(0)
            # Quantize ke bytes
            raw_bytes = (raw.cpu().numpy() * 255).astype(np.uint8).tobytes()
            # HKDF key derivation — output selalu 32 bytes (AES-256)
            key = HKDF(raw_bytes, 32, salt, SHA256, 1)
        return key


# Buat model
model = NeuralKeyGen(
    input_dim  = CFG["FEATURE_DIM"],
    hidden1    = CFG["HIDDEN_1"],
    hidden2    = CFG["HIDDEN_2"],
    output_dim = CFG["OUTPUT_DIM"],
    dropout    = CFG["DROPOUT"],
).to(device)

# Hitung parameter
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✓ Model NeuralKeyGen berhasil dibuat")
print(f"  Total parameter   : {total_params:,}")
print(f"  Trainable params  : {trainable:,}")
print(f"  Model size (est.) : {total_params * 4 / 1e6:.2f} MB")
print()
print(model)


## 7. Loss Function Khusus NeuralKeyGen

In [ ]:
# ─── CUSTOM LOSS FUNCTION ────────────────────────────────────────────────────

class NeuralKeyGenLoss(nn.Module):
    """
    Triplet-based loss dengan 3 komponen:

    1. Consistency Loss  (bobot 0.5)
       anchor dan positive (input sama dgn noise kecil) harus output IDENTIK.
       Menggunakan MSE.

    2. Separation Loss   (bobot 0.3)
       anchor dan negative (input berbeda) harus output SANGAT BERBEDA.
       Menggunakan cosine similarity — ingin nilai mendekati -1 atau 0.

    3. Entropy Loss      (bobot 0.2)
       Output harus terdistribusi merata [0, 1], tidak bias ke 0 atau 1.
       Menggunakan binary cross-entropy dengan target 0.5.
    """

    def __init__(self, w_cons: float, w_sep: float, w_ent: float,
                 sep_margin: float = 0.2):
        super().__init__()
        self.w_cons = w_cons
        self.w_sep  = w_sep
        self.w_ent  = w_ent
        self.margin = sep_margin

    def forward(self, out_anchor, out_positive, out_negative):
        # 1. Consistency: anchor vs positive harus sama
        consistency = F.mse_loss(out_anchor, out_positive)

        # 2. Separation: anchor vs negative harus berbeda
        # cosine_similarity ∈ [-1, 1]; kita ingin < margin (jauh)
        cos_sim = F.cosine_similarity(out_anchor, out_negative, dim=1)
        separation = torch.mean(F.relu(cos_sim - self.margin + 0.1))

        # 3. Entropy: output mendekati uniform (mean ~0.5)
        target = torch.full_like(out_anchor, 0.5)
        entropy_loss = F.binary_cross_entropy(out_anchor, target)

        total = (self.w_cons * consistency +
                 self.w_sep  * separation  +
                 self.w_ent  * entropy_loss)

        return total, {
            'consistency': consistency.item(),
            'separation' : separation.item(),
            'entropy'    : entropy_loss.item(),
        }


criterion = NeuralKeyGenLoss(
    w_cons = CFG["W_CONSISTENCY"],
    w_sep  = CFG["W_SEPARATION"],
    w_ent  = CFG["W_ENTROPY"],
)

# Test loss dengan batch dummy
with torch.no_grad():
    dummy_a = torch.rand(8, CFG["OUTPUT_DIM"])
    dummy_p = dummy_a + torch.randn_like(dummy_a) * 0.01
    dummy_n = torch.rand(8, CFG["OUTPUT_DIM"])
    loss, breakdown = criterion(dummy_a, dummy_p, dummy_n)

print("✓ Loss function siap")
print(f"  Loss dummy test      : {loss.item():.4f}")
print(f"  - Consistency        : {breakdown['consistency']:.4f}")
print(f"  - Separation         : {breakdown['separation']:.4f}")
print(f"  - Entropy            : {breakdown['entropy']:.4f}")


## 8. Training Loop

In [ ]:
# ─── TRAINING LOOP ───────────────────────────────────────────────────────────

optimizer = AdamW(model.parameters(),
                  lr=CFG["LR"],
                  weight_decay=CFG["WEIGHT_DECAY"])

scheduler = CosineAnnealingLR(optimizer,
                               T_max=CFG["EPOCHS"],
                               eta_min=CFG["LR"] * 0.01)

# History
history = {
    'train_loss': [], 'val_loss': [],
    'consistency': [], 'separation': [], 'entropy': [],
    'lr': []
}

best_val_loss = float('inf')
patience_counter = 0
best_model_path = f"{CFG['SAVE_DIR']}/{CFG['MODEL_NAME']}"

print(f"Training NeuralKeyGen — {CFG['EPOCHS']} epochs")
print(f"{'Epoch':>5} {'Train':>8} {'Val':>8} {'Cons':>8} {'Sep':>8} {'Ent':>8} {'LR':>8} {'Time':>6}")
print("-" * 65)

total_start = time.time()

for epoch in range(1, CFG["EPOCHS"] + 1):
    ep_start = time.time()

    # ── TRAIN ──────────────────────────────────────────────────────────────
    model.train()
    train_losses = []
    breakdown_sums = {'consistency': 0., 'separation': 0., 'entropy': 0.}

    for anchor, positive, negative in train_loader:
        anchor   = anchor.to(device, non_blocking=True)
        positive = positive.to(device, non_blocking=True)
        negative = negative.to(device, non_blocking=True)

        out_a = model(anchor)
        out_p = model(positive)
        out_n = model(negative)

        loss, bkd = criterion(out_a, out_p, out_n)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_losses.append(loss.item())
        for k in breakdown_sums:
            breakdown_sums[k] += bkd[k]

    train_loss = np.mean(train_losses)
    for k in breakdown_sums:
        breakdown_sums[k] /= len(train_loader)

    # ── VALIDATE ───────────────────────────────────────────────────────────
    model.eval()
    val_losses = []
    with torch.no_grad():
        for anchor, positive, negative in val_loader:
            anchor   = anchor.to(device, non_blocking=True)
            positive = positive.to(device, non_blocking=True)
            negative = negative.to(device, non_blocking=True)
            out_a = model(anchor)
            out_p = model(positive)
            out_n = model(negative)
            loss, _ = criterion(out_a, out_p, out_n)
            val_losses.append(loss.item())

    val_loss = np.mean(val_losses)
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    ep_time = time.time() - ep_start

    # Simpan history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['consistency'].append(breakdown_sums['consistency'])
    history['separation'].append(breakdown_sums['separation'])
    history['entropy'].append(breakdown_sums['entropy'])
    history['lr'].append(current_lr)

    # Log setiap epoch
    print(f"{epoch:>5} {train_loss:>8.4f} {val_loss:>8.4f} "
          f"{breakdown_sums['consistency']:>8.4f} "
          f"{breakdown_sums['separation']:>8.4f} "
          f"{breakdown_sums['entropy']:>8.4f} "
          f"{current_lr:>8.2e} {ep_time:>5.1f}s")

    # Checkpoint terbaik
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch'     : epoch,
            'model_state': model.state_dict(),
            'optimizer' : optimizer.state_dict(),
            'val_loss'  : best_val_loss,
            'config'    : CFG,
        }, best_model_path)
        print(f"       ↑ Model terbaik disimpan (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= CFG["PATIENCE"]:
            print(f"\nEarly stopping pada epoch {epoch} (patience={CFG['PATIENCE']})")
            break

total_time = time.time() - total_start
print(f"\n✓ Training selesai dalam {total_time/60:.1f} menit")
print(f"  Best val loss : {best_val_loss:.4f}")
print(f"  Model disimpan: {best_model_path}")


## 9. Visualisasi Training

In [ ]:
# ─── PLOT TRAINING HISTORY ───────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("NeuralKeyGen — Training History", fontsize=13, fontweight='bold')

epochs_ran = len(history['train_loss'])
ep_range = range(1, epochs_ran + 1)

# 1. Loss keseluruhan
ax = axes[0, 0]
ax.plot(ep_range, history['train_loss'], label='Train', color='#2563EB', linewidth=1.5)
ax.plot(ep_range, history['val_loss'],   label='Val',   color='#DC2626', linewidth=1.5)
ax.set_title("Total Loss")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Komponen loss
ax = axes[0, 1]
ax.plot(ep_range, history['consistency'], label='Consistency', color='#16A34A', linewidth=1.5)
ax.plot(ep_range, history['separation'],  label='Separation',  color='#D97706', linewidth=1.5)
ax.plot(ep_range, history['entropy'],     label='Entropy',     color='#7C3AED', linewidth=1.5)
ax.set_title("Komponen Loss")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Learning rate
ax = axes[1, 0]
ax.plot(ep_range, history['lr'], color='#0891B2', linewidth=1.5)
ax.set_title("Learning Rate (Cosine Annealing)")
ax.set_xlabel("Epoch")
ax.set_ylabel("LR")
ax.grid(True, alpha=0.3)

# 4. Train vs Val gap
ax = axes[1, 1]
gap = [abs(t - v) for t, v in zip(history['train_loss'], history['val_loss'])]
ax.fill_between(ep_range, 0, gap, alpha=0.4, color='#DC2626')
ax.plot(ep_range, gap, color='#DC2626', linewidth=1)
ax.set_title("Generalization Gap (|Train - Val|)")
ax.set_xlabel("Epoch")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CFG['SAVE_DIR']}/training_history.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"✓ Plot disimpan")


## 10. Evaluasi Model — 4 Properti Kunci

In [ ]:
# ─── EVALUASI LENGKAP ────────────────────────────────────────────────────────

# Load model terbaik
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"✓ Model terbaik dimuat (epoch={checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f})")

# Ambil sample test
test_features = X_test[:1000]
test_tensor = torch.tensor(test_features, dtype=torch.float32).to(device)

with torch.no_grad():
    test_outputs = model(test_tensor).cpu().numpy()

print(f"\n{'='*55}")
print("EVALUASI 4 PROPERTI KRITIS NeuralKeyGen")
print(f"{'='*55}")

# ── PROPERTI 1: DETERMINISTIK ────────────────────────────────────────────────
print("\n1. DETERMINISTIK")
n_repeat = 5
outputs_repeated = []
for _ in range(n_repeat):
    with torch.no_grad():
        out = model(test_tensor[:100]).cpu().numpy()
    outputs_repeated.append(out)

max_diff = max(
    np.max(np.abs(outputs_repeated[i] - outputs_repeated[0]))
    for i in range(1, n_repeat)
)
is_deterministic = max_diff < 1e-6
print(f"   Max perbedaan dari {n_repeat} run : {max_diff:.2e}")
print(f"   Status: {'LULUS ✓' if is_deterministic else 'GAGAL ✗'}")

# ── PROPERTI 2: COLLISION RESISTANCE ─────────────────────────────────────────
print("\n2. COLLISION RESISTANCE")
n_pairs = 50_000
idx1 = np.random.choice(len(test_outputs), n_pairs)
idx2 = np.random.choice(len(test_outputs), n_pairs)
mask = idx1 != idx2
idx1, idx2 = idx1[mask], idx2[mask]

# Quantize ke bytes (seperti produksi)
quantized = (test_outputs * 255).astype(np.uint8)
collisions = np.sum(np.all(quantized[idx1] == quantized[idx2], axis=1))
collision_rate = collisions / len(idx1)
print(f"   Pasangan uji         : {len(idx1):,}")
print(f"   Collision ditemukan  : {collisions}")
print(f"   Collision rate       : {collision_rate:.2e}")
print(f"   Status: {'LULUS ✓' if collisions == 0 else 'PERLU REVIEW !'}")

# ── PROPERTI 3: SEPARATION (HAMMING DISTANCE) ─────────────────────────────────
print("\n3. SEPARATION — Hamming Distance antar kunci")
n_sample_pairs = 5000
i1 = np.random.choice(len(quantized), n_sample_pairs)
i2 = np.random.choice(len(quantized), n_sample_pairs)
same_mask = i1 != i2
i1, i2 = i1[same_mask], i2[same_mask]

hamming_dists = []
for a, b in zip(quantized[i1[:1000]], quantized[i2[:1000]]):
    bits_a = np.unpackbits(a)
    bits_b = np.unpackbits(b)
    hamming_dists.append(np.sum(bits_a != bits_b))

hamming_arr = np.array(hamming_dists)
print(f"   Mean Hamming distance : {hamming_arr.mean():.1f} bit (dari {CFG['OUTPUT_DIM']} bit)")
print(f"   Min Hamming distance  : {hamming_arr.min()} bit")
print(f"   Std Hamming distance  : {hamming_arr.std():.1f} bit")
print(f"   Ideal target          : ~128 bit (50% dari {CFG['OUTPUT_DIM']})")
strong_sep = hamming_arr.mean() > 80
print(f"   Status: {'LULUS ✓' if strong_sep else 'LEMAH — perlu lebih training'}")

# ── PROPERTI 4: ENTROPY / DISTRIBUSI OUTPUT ──────────────────────────────────
print("\n4. ENTROPY OUTPUT")
output_mean = test_outputs.mean()
output_std  = test_outputs.std()
output_min  = test_outputs.min()
output_max  = test_outputs.max()

# Ideal: mean ≈ 0.5, std ≈ 0.25 (distribusi uniform)
entropy_ok = 0.4 < output_mean < 0.6 and output_std > 0.15
print(f"   Mean output  : {output_mean:.4f} (ideal ~0.5)")
print(f"   Std output   : {output_std:.4f} (ideal >0.15)")
print(f"   Min output   : {output_min:.4f}")
print(f"   Max output   : {output_max:.4f}")
print(f"   Status: {'LULUS ✓' if entropy_ok else 'PERLU TUNING'}")

print(f"\n{'='*55}")
all_pass = is_deterministic and (collisions == 0) and strong_sep and entropy_ok
print(f"HASIL AKHIR: {'SEMUA LULUS — Model siap produksi ✓' if all_pass else 'ADA YANG PERLU DIPERBAIKI'}")
print(f"{'='*55}")


## 11. Visualisasi Evaluasi

In [ ]:
# ─── PLOT EVALUASI ───────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("NeuralKeyGen — Evaluasi Properti Kunci", fontsize=13, fontweight='bold')

# 1. Distribusi output values
ax = axes[0, 0]
ax.hist(test_outputs.flatten(), bins=80, color='#4A90D9',
        edgecolor='none', density=True)
ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='Ideal mean=0.5')
ax.axvline(test_outputs.mean(), color='orange', linestyle='--',
           linewidth=1.5, label=f'Actual mean={test_outputs.mean():.3f}')
ax.set_title("Distribusi nilai output (semua bit)")
ax.set_xlabel("Nilai (0-1)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Hamming distance distribution
ax = axes[0, 1]
ax.hist(hamming_dists, bins=40, color='#16A34A', edgecolor='none', density=True)
ax.axvline(np.mean(hamming_dists), color='red', linestyle='--',
           linewidth=1.5, label=f'Mean={np.mean(hamming_dists):.0f} bit')
ax.axvline(128, color='gray', linestyle=':', linewidth=1.5, label='Ideal=128 bit')
ax.set_title("Distribusi Hamming Distance antar kunci")
ax.set_xlabel("Hamming distance (bit)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. Cosine similarity antar output berbeda
ax = axes[1, 0]
sample_outs = test_outputs[:500]
cos_sims = []
for i in range(0, 500, 2):
    a = sample_outs[i]
    b = sample_outs[i+1]
    cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
    cos_sims.append(cos)
ax.hist(cos_sims, bins=40, color='#D97706', edgecolor='none', density=True)
ax.axvline(0, color='gray', linestyle=':', linewidth=1.5, label='Ideal cos_sim≈0')
ax.axvline(np.mean(cos_sims), color='red', linestyle='--',
           linewidth=1.5, label=f'Mean={np.mean(cos_sims):.3f}')
ax.set_title("Cosine similarity antar kunci berbeda")
ax.set_xlabel("Cosine similarity")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 4. Heatmap output sample 10 siswa
ax = axes[1, 1]
sample_10 = test_outputs[:10, :64]  # tampilkan 64 bit pertama
im = ax.imshow(sample_10, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set_title("Output 10 siswa pertama (64 bit pertama)")
ax.set_xlabel("Bit ke-")
ax.set_ylabel("Siswa ke-")
ax.set_yticks(range(10))
ax.set_yticklabels([f"Siswa {i+1}" for i in range(10)], fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig(f"{CFG['SAVE_DIR']}/evaluation.png", dpi=120, bbox_inches='tight')
plt.show()


## 12. Inference Produksi — Cara Pakai di Aplikasi FastAPI

In [ ]:
# ─── PRODUCTION INFERENCE ────────────────────────────────────────────────────

class NeuralKeyGenInference:
    """
    Class ini yang dipakai di aplikasi produksi (FastAPI).
    Cara pakai: import class ini dan panggil generate_key().
    """

    def __init__(self, model_path: str, system_salt: bytes, device_str: str = 'cpu'):
        self.device = torch.device(device_str)
        self.system_salt = system_salt
        self.extractor = StudentFeatureExtractor(system_salt)

        # Load model
        self.model = NeuralKeyGen(
            input_dim  = CFG["FEATURE_DIM"],
            hidden1    = CFG["HIDDEN_1"],
            hidden2    = CFG["HIDDEN_2"],
            output_dim = CFG["OUTPUT_DIM"],
            dropout    = CFG["DROPOUT"],
        ).to(self.device)

        ckpt = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state'])
        self.model.eval()
        print(f"  Model loaded from epoch {ckpt['epoch']}")

    def generate_key(self, siswa_profile: dict) -> bytes:
        """
        Input  : dict profil siswa (dari database)
        Output : 32 bytes = kunci AES-256 siap pakai
        
        Kunci ini TIDAK disimpan — dibuat ulang setiap kali dari input yang sama.
        """
        features = self.extractor.extract(siswa_profile)
        x = torch.tensor(features, dtype=torch.float32).unsqueeze(0).to(self.device)

        with torch.no_grad():
            raw_output = self.model(x).squeeze(0).cpu().numpy()

        # Quantize ke bytes
        raw_bytes = (raw_output * 255).astype(np.uint8).tobytes()

        # HKDF derivation → 32 bytes AES-256 key
        aes_key = HKDF(
            master    = raw_bytes,
            key_len   = 32,
            salt      = self.system_salt,
            hashmod   = SHA256,
            num_keys  = 1,
        )
        return aes_key

    def verify_deterministic(self, siswa_profile: dict, n_runs: int = 10) -> bool:
        """Verifikasi bahwa kunci selalu sama untuk input yang sama."""
        keys = [self.generate_key(siswa_profile) for _ in range(n_runs)]
        return all(k == keys[0] for k in keys)


# ─── TEST PRODUKSI ───────────────────────────────────────────────────────────

print("=" * 55)
print("TEST PRODUKSI — Simulasi alur aplikasi nyata")
print("=" * 55)

nkg = NeuralKeyGenInference(
    model_path  = best_model_path,
    system_salt = CFG["SYSTEM_SALT"],
    device_str  = 'cpu',  # CPU di produksi lebih stabil
)

# Profil siswa contoh (seperti yang ada di database)
siswa_budi = {
    'nis'           : '20240001',
    'tahun_masuk'   : 2024,
    'npsn'          : 'NPSN00001',
    'nama_hash'     : hashlib.sha256('Budi Santoso'.encode()).hexdigest(),
    'tgl_lahir'     : '2008-03-15',
    'entropy_seed'  : 'a3f8b2c1d9e4f7a6b5c8d2e1f9a4b7c3',
    'reg_timestamp' : 1700000001,
}

siswa_rika = {
    'nis'           : '20240002',
    'tahun_masuk'   : 2024,
    'npsn'          : 'NPSN00001',
    'nama_hash'     : hashlib.sha256('Rika Dewi'.encode()).hexdigest(),
    'tgl_lahir'     : '2008-07-22',
    'entropy_seed'  : 'f1e2d3c4b5a6978869504132210fedcb',
    'reg_timestamp' : 1700000002,
}

print("\n[Budi Santoso]")
key_budi_1 = nkg.generate_key(siswa_budi)
key_budi_2 = nkg.generate_key(siswa_budi)
key_budi_3 = nkg.generate_key(siswa_budi)
print(f"  Run 1: {key_budi_1.hex()}")
print(f"  Run 2: {key_budi_2.hex()}")
print(f"  Run 3: {key_budi_3.hex()}")
print(f"  Deterministik: {'YA ✓' if key_budi_1 == key_budi_2 == key_budi_3 else 'TIDAK ✗'}")

print("\n[Rika Dewi]")
key_rika = nkg.generate_key(siswa_rika)
print(f"  Key  : {key_rika.hex()}")

print("\n[Perbandingan]")
print(f"  Budi key : {key_budi_1.hex()}")
print(f"  Rika key : {key_rika.hex()}")
print(f"  Sama?    : {'YA (BUG!)' if key_budi_1 == key_rika else 'TIDAK ✓ — kunci unik per siswa'}")

bits_budi = np.unpackbits(np.frombuffer(key_budi_1, dtype=np.uint8))
bits_rika = np.unpackbits(np.frombuffer(key_rika,   dtype=np.uint8))
hamming_prod = np.sum(bits_budi != bits_rika)
print(f"  Hamming distance: {hamming_prod} bit dari 256 bit")

det_ok = nkg.verify_deterministic(siswa_budi, n_runs=20)
print(f"\n  Deterministic test (20 runs): {'LULUS ✓' if det_ok else 'GAGAL ✗'}")


## 13. Export Model & Download

In [ ]:
# ─── EXPORT & PACKAGE ────────────────────────────────────────────────────────
import shutil, zipfile

save_dir = Path(CFG["SAVE_DIR"])

# Simpan konfigurasi model
cfg_export = {k: v for k, v in CFG.items() if k != 'SYSTEM_SALT'}
cfg_export['SYSTEM_SALT'] = CFG['SYSTEM_SALT'].decode()
with open(save_dir / 'config.json', 'w') as f:
    json.dump(cfg_export, f, indent=2)

# Simpan metadata evaluasi
eval_meta = {
    'deterministic'      : bool(is_deterministic),
    'collisions_found'   : int(collisions),
    'collision_rate'     : float(collision_rate),
    'mean_hamming_dist'  : float(np.mean(hamming_dists)),
    'output_mean'        : float(output_mean),
    'output_std'         : float(output_std),
    'best_val_loss'      : float(best_val_loss),
    'epochs_trained'     : len(history['train_loss']),
    'n_parameters'       : trainable,
    'n_training_samples' : len(X_train),
}
with open(save_dir / 'eval_metrics.json', 'w') as f:
    json.dump(eval_meta, f, indent=2)

# Simpan versi torchscript (lebih cepat di produksi)
model.eval()
example_input = torch.rand(1, CFG["FEATURE_DIM"])
try:
    scripted = torch.jit.trace(model.cpu(), example_input)
    scripted.save(str(save_dir / "neuralkeygen_scripted.pt"))
    print("✓ TorchScript model disimpan")
except Exception as e:
    print(f"  TorchScript skip: {e}")

# ZIP semua output
zip_path = "/content/NeuralKeyGen_Model.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in save_dir.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(save_dir.parent))

zip_size = Path(zip_path).stat().st_size / 1e6
print(f"✓ Package siap: {zip_path}")
print(f"  Ukuran ZIP : {zip_size:.1f} MB")
print()
print("File dalam package:")
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f"  {name:<50} {info.file_size/1e3:>8.1f} KB")

# Download otomatis di Colab
try:
    from google.colab import files
    files.download(zip_path)
    print("\n✓ Download dimulai otomatis!")
except Exception:
    print(f"\n  (Bukan di Colab — file ada di: {zip_path})")

print()
print("=" * 55)
print("RINGKASAN AKHIR")
print("=" * 55)
print(f"  Model     : NeuralKeyGen v1.0")
print(f"  Input dim : {CFG['FEATURE_DIM']}")
print(f"  Output    : {CFG['OUTPUT_DIM']}-bit → 32 bytes AES-256 key")
print(f"  Params    : {trainable:,}")
print(f"  Val loss  : {best_val_loss:.4f}")
for k, v in eval_meta.items():
    if isinstance(v, float):
        print(f"  {k:<25}: {v:.4f}")
    else:
        print(f"  {k:<25}: {v}")


## 14. Kode Integrasi FastAPI (Copy ke Aplikasi)

Salin kode di bawah ke file `backend/app/core/neural_key.py` di proyek FastAPI Anda.


In [ ]:
# ─── KODE INTEGRASI FASTAPI ──────────────────────────────────────────────────
# Simpan sebagai: backend/app/core/neural_key.py

fastapi_code = '''
"""
neural_key.py — NeuralKeyGen integration untuk FastAPI DMS Sekolah
"""
import hashlib
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from functools import lru_cache
from Crypto.Protocol.KDF import HKDF
from Crypto.Hash import SHA256


# ── Model definition (sama persis dengan training) ──────────────────────────

class NeuralKeyGen(nn.Module):
    def __init__(self, input_dim=64, hidden1=512, hidden2=256,
                 output_dim=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.BatchNorm1d(hidden1),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2), nn.BatchNorm1d(hidden2),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden2, hidden2), nn.BatchNorm1d(hidden2),
            nn.ReLU(inplace=True), nn.Dropout(dropout * 0.5),
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden2, output_dim), nn.Sigmoid()
        )

    def forward(self, x):
        return self.output_head(self.encoder(x))


# ── Key generator singleton ──────────────────────────────────────────────────

@lru_cache(maxsize=1)
def get_key_generator():
    """Singleton — model dimuat sekali saat startup FastAPI."""
    return NeuralKeyGenService(
        model_path  = "ml_model/neuralkeygen.pt",
        system_salt = b"DMS_SEKOLAH_SALT_v1_2025",  # dari .env
    )


class NeuralKeyGenService:
    def __init__(self, model_path: str, system_salt: bytes):
        self.salt   = system_salt
        self.device = torch.device("cpu")  # CPU lebih stabil untuk inference

        self.model = NeuralKeyGen().to(self.device)
        ckpt = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()

    def _extract_features(self, siswa: dict) -> torch.Tensor:
        """Sama dengan StudentFeatureExtractor di notebook training."""
        import numpy as np
        from datetime import datetime

        def sinusoidal(val, n, scale=10000):
            d = n // 2
            return np.array(
                [np.sin(val/(scale**(2*i/n))) for i in range(d)] +
                [np.cos(val/(scale**(2*i/n))) for i in range(d)], dtype=np.float32)

        def cyclical(val, period):
            a = 2 * np.pi * val / period
            return np.array([np.sin(a), np.cos(a)], dtype=np.float32)

        def hash_float(text, n):
            h = hashlib.sha256(text.encode()).digest()
            return np.frombuffer(h[:n], dtype=np.uint8).astype(np.float32) / 255.0

        f = []
        nis_val = int(str(siswa["nis"]).replace("-", "")) % 100_000_000
        f.append(sinusoidal(nis_val, 8, 10_000_000))

        tahun = int(siswa["tahun_masuk"])
        f.append(cyclical(tahun - 2010, 20))
        f.append(np.array([(tahun-2010)/20.0, float(tahun%2)], dtype=np.float32))

        f.append(hash_float(siswa["npsn"], 4))
        f.append(hash_float(siswa["nama_hash"], 8))

        tgl = datetime.strptime(siswa["tgl_lahir"], "%Y-%m-%d")
        f.append(cyclical(tgl.day, 31))
        f.append(cyclical(tgl.month, 12))
        f.append(cyclical(tgl.year - 1995, 30))
        f.append(np.array([float(tgl.weekday())/6.0, float(tgl.month<=6)], dtype=np.float32))

        seed_bytes = bytes.fromhex(siswa["entropy_seed"])
        f.append(np.frombuffer(seed_bytes, dtype=np.uint8).astype(np.float32) / 255.0)

        ts = int(siswa.get("reg_timestamp", 1_700_000_000))
        f.append(cyclical(ts % 86400, 86400))
        f.append(cyclical((ts // 86400) % 365, 365))

        f.append(hash_float(siswa["npsn"] + siswa["entropy_seed"][:8], 4))

        nis_val2 = nis_val * tgl.month * tgl.day
        f.append(sinusoidal(nis_val2 % 1_000_000, 4, 100_000))
        f.append(np.zeros(4, dtype=np.float32))

        vec = np.concatenate(f)
        return torch.tensor(vec, dtype=torch.float32).unsqueeze(0)

    def generate_key(self, siswa: dict) -> bytes:
        """Hasilkan AES-256 key dari profil siswa. Kunci tidak disimpan."""
        x = self._extract_features(siswa).to(self.device)
        with torch.no_grad():
            raw = self.model(x).squeeze(0).cpu().numpy()
        raw_bytes = (raw * 255).astype(np.uint8).tobytes()
        return HKDF(raw_bytes, 32, self.salt, SHA256, 1)


# ── FastAPI endpoint contoh ──────────────────────────────────────────────────

# from fastapi import Depends
# from app.core.neural_key import get_key_generator
#
# @router.get("/dokumen/{doc_id}/download")
# async def download_document(
#     doc_id: int,
#     current_user = Depends(get_current_user),
#     nkg = Depends(get_key_generator),
# ):
#     # Ambil profil siswa dari DB
#     siswa = await db.get_siswa(current_user.id)
#
#     # Generate kunci — tidak ada di DB, dibuat real-time
#     aes_key = nkg.generate_key(siswa.to_dict())
#
#     # Dekripsi dokumen
#     encrypted_doc = await storage.get(doc.file_path)
#     decrypted_doc = AES256_GCM.decrypt(encrypted_doc, aes_key)
#
#     # Tambah watermark
#     final_pdf = add_watermark(decrypted_doc, siswa.nama, datetime.now())
#     return StreamingResponse(final_pdf, media_type="application/pdf")
'''

output_path = save_dir / "neural_key_fastapi.py"
with open(output_path, 'w') as f:
    f.write(fastapi_code.strip())

print(f"✓ Kode integrasi FastAPI disimpan di:")
print(f"  {output_path}")
print()
print("Cara pakai di proyek FastAPI:")
print("  1. Salin neural_key_fastapi.py ke backend/app/core/neural_key.py")
print("  2. Salin neuralkeygen.pt ke backend/ml_model/")
print("  3. pip install pycryptodome torch")
print("  4. Import: from app.core.neural_key import get_key_generator")
